# Data Cleaning and Feature Engineering


In [ ]:
import sys, os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / '.git').exists() or (candidate / 'pyproject.toml').exists():
            return candidate
    return start.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from sqlalchemy import text

from app.core.database import engine

with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
print('Database connection verified')
print(f'Project root: {PROJECT_ROOT}')

## 1. Load Market Prices
Wholesale cereal prices for Maize, Millet, and Sorghum across five markets — Tamale, Bolga, and Wa as the primary northern targets; Kumasi and Techiman as southern reference markets. EDA (notebook 03) found no detectable lead-lag between southern and northern markets at monthly resolution — all markets move simultaneously (r > 0.98). Southern markets are included as spatial price signals, not as leading indicators.

In [ ]:
query = """
    SELECT
        p.date,
        p.market,
        p.market_id,
        p.commodity,
        p.price,
        p.pricetype,
        m.latitude,
        m.longitude,
        m.admin1
    FROM wfp_prices p
    JOIN wfp_markets m ON p.market_id = m.market_id
    WHERE p.commodity IN ('Maize', 'Millet', 'Sorghum')
    AND p.pricetype = 'Wholesale'
    AND p.market IN ('Tamale', 'Bolga', 'Wa', 'Kumasi', 'Techiman')
    ORDER BY p.market, p.commodity, p.date
"""

df = pd.read_sql(query, engine)
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

print(f"total records loaded: {len(df)}")
print(f"markets: {sorted(df['market'].unique())}")
print(f"commodities: {sorted(df['commodity'].unique())}")
print(f"date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"missing values: {df.isnull().sum().sum()}")
print()
print("records per market per commodity:")
print(df.groupby(['market', 'commodity']).size().unstack(fill_value=0))

## 2. Exchange Rate
Monthly GHS/USD rate from FAOSTAT merged via `merge_asof` (each price record gets the closest preceding monthly FX reading). Using monthly rather than annual values preserves within-year variation — critical for capturing the 2022–2023 cedi collapse, where the rate moved sharply mid-year.

In [ ]:
fx_query = """
    SELECT year, months, value as exchange_rate
    FROM ghana_exchange_rates
    WHERE months != 'Annual value'
    AND element = 'Local currency units per USD'
    ORDER BY year, months
"""

fx = pd.read_sql(fx_query, engine)
fx['year'] = fx['year'].astype(int)

# No redenomination adjustment needed: FAOSTAT monthly values are already in
# new Ghana cedis (GHS) for all years including 2006-2007 (verified: Jan 2006
# = 0.908 GHS/USD, which is correct). The annual rows for 2006-2007 are in old
# cedis, but we exclude those above with months != 'Annual value'.
fx['date'] = pd.to_datetime(fx['year'].astype(str) + ' ' + fx['months'], format='%Y %B')
fx = fx[['date', 'exchange_rate']].sort_values('date').reset_index(drop=True)

# merge_asof: each price record gets the closest preceding monthly FX reading
# this preserves within-year variation — critical for the 2022-23 cedi collapse
df = df.sort_values('date').reset_index(drop=True)
df = pd.merge_asof(df, fx, on='date')
df = df.drop_duplicates(subset=['date', 'market', 'commodity']).reset_index(drop=True)

print(f"FX records (monthly): {len(fx)}")
print(f"FX date range: {fx['date'].min().date()} → {fx['date'].max().date()}")
print(f"rows after merge: {len(df)}")
print(f"missing exchange rates: {df['exchange_rate'].isnull().sum()}")
print()
print("sample — 2006 and 2007 values should be ~0.9 GHS/USD (new cedi):")
print(fx[fx['date'].dt.year.isin([2006, 2007])].head(6).to_string(index=False))

## 3. Producer Price Index
Farm gate price index (2014–2016 = 100) from FAOSTAT merged by year and commodity. Captures the distress-selling spread: when farm gate prices sit far below market prices, distress selling is active and the subsequent recovery tends to be stronger.

In [ ]:
# FAOSTAT has no maize producer prices for Ghana — millet/sorghum average used as proxy
pp_query = """
    SELECT year, item, value as producer_price_index
    FROM fao_producer_prices
    WHERE item IN ('Maize', 'Millet', 'Sorghum')
    AND months = 'Annual value'
    AND element = 'Producer Price Index (2014-2016 = 100)'
    ORDER BY item, year
"""

pp = pd.read_sql(pp_query, engine)
pp['year'] = pp['year'].astype(int)
pp = pp[pp['producer_price_index'] < 10000].copy()
pp = pp.groupby(['year', 'item'])['producer_price_index'].mean().reset_index()

pp_wide = pp.pivot(index='year', columns='item', values='producer_price_index').reset_index()
pp_wide.columns.name = None

# skipna=False requires both Millet and Sorghum to exist before computing the proxy
maize_proxy = pp_wide[['Millet', 'Sorghum']].mean(axis=1, skipna=False)
if 'Maize' in pp_wide.columns:
    pp_wide['Maize'] = pp_wide['Maize'].fillna(maize_proxy)
else:
    pp_wide['Maize'] = maize_proxy

print('producer price table from 2006 onwards:')
print(pp_wide[pp_wide['year'] >= 2006].to_string(index=False))

pp_long = pp_wide.melt(id_vars='year', var_name='commodity', value_name='producer_price_index')
df['year'] = df['date'].dt.year
df = df.merge(pp_long, on=['year', 'commodity'], how='left').drop(columns='year')

print(f'\nrows after producer price merge: {len(df)}')
print(f"missing producer index: {df['producer_price_index'].isnull().sum()}")
print()
print(df[['date', 'market', 'commodity', 'price', 'producer_price_index']].head(8))

## 4. Outlier Removal
IQR filter applied independently per market-commodity group to remove data entry errors while preserving genuine price extremes driven by the 2022–2023 currency shock.

In [ ]:
# IQR multiplier of 5 preserves genuine 2022-2023 cedi-collapse prices that multiplier 3 removed
n_before_outlier = len(df)
print(f"records before outlier removal: {n_before_outlier}")

def iqr_mask(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return series.between(q1 - 5 * iqr, q3 + 5 * iqr)

mask = df.groupby(["market", "commodity"])["price"].transform(iqr_mask)
df = df.loc[mask].reset_index(drop=True)

print(f"records after outlier removal: {len(df)}")
print(f"records removed: {n_before_outlier - len(df)}")
print()
print("records per market per commodity after cleaning:")
print(df.groupby(["market", "commodity"]).size().unstack(fill_value=0))
print()
print(f"2023 records kept: {len(df[df['date'].dt.year == 2023])}")
print(f"date range: {df['date'].min().date()} to {df['date'].max().date()}")

## 5. Feature Engineering
Month encoded as sin/cos so the model treats December and January as adjacent.

In [ ]:
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

print(f"month_sin range: {df['month_sin'].min():.2f} to {df['month_sin'].max():.2f}")
print(f"month_cos range: {df['month_cos'].min():.2f} to {df['month_cos'].max():.2f}")
print()
print(df[['date', 'market', 'commodity', 'price', 'month_sin', 'month_cos']].head(6))

## 6. Save Outputs
Saves the clean dataset with raw numeric values.

In [ ]:
final_cols = [
    'date', 'market', 'commodity', 'price',
    'latitude', 'longitude',
    'exchange_rate', 'producer_price_index',
    'month_sin', 'month_cos',
]

df_clean = df[final_cols].copy()

n_before_drop = len(df_clean)
df_clean = df_clean.dropna().reset_index(drop=True)
print(f"rows dropped by dropna (missing fx or producer index): {n_before_drop - len(df_clean)}")

os.makedirs('data/processed', exist_ok=True)
df_clean.to_csv('data/processed/cereals_merged_clean.csv', index=False)

print()
print(f"total rows: {len(df_clean)}")
print(f"columns: {list(df_clean.columns)}")
print(f"markets: {sorted(df_clean['market'].unique())}")
print(f"commodities: {sorted(df_clean['commodity'].unique())}")
print(f"date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"missing values: {df_clean.isnull().sum().sum()}")
print()
print("records per market per commodity:")
print(df_clean.groupby(['market', 'commodity']).size().unstack(fill_value=0))
print()
print(f"clean csv exists: {os.path.exists('data/processed/cereals_merged_clean.csv')}")